# 주택 가격 데이터 처리 — 과제 1-1 ~ 1-5

**제출용 점검본** · Python 기초 과제  
사용 데이터: `1.01.house.price.csv` (10행, 4열)

- 각 문제를 위에서 아래로 실행합니다.
- 정상 파일·없는 파일 결과, 유효하지 않은 입력의 처리 결과, 함수 테스트 7건을 출력합니다.
- CSV 파일은 노트북과 같은 폴더에 두거나 Colab의 `/content`에 업로드합니다.

> 이 과제는 **파일 읽기, 반복문, 조건문, 함수, 예외 처리, 자료구조**를 연습하는 과제입니다. 회귀 모델이나 교차검증은 사용하지 않습니다.

## 1-1. CSV 파일 읽기

**확인:** 첫 행, 전체 행 수, 첫 행 `price`의 자료형을 출력하고 없는 파일도 테스트합니다.

In [1]:
from pathlib import Path
import math
import pandas as pd

# 로컬 Jupyter 및 Colab의 CSV 경로 지원
csv_name = '1.01.house.price.csv'
csv_path = Path(csv_name)
if not csv_path.exists() and Path('/content', csv_name).exists():
    csv_path = Path('/content', csv_name)


def load_house_data(file_path):
    try:
        df = pd.read_csv(file_path)
    except FileNotFoundError:
        print('파일명과 업로드 여부를 확인하세요.')
        return None

    # 성공한 경우에만 데이터 확인
    print('첫 번째 행:')
    print(df.iloc[0])
    print('전체 데이터 행 수:', len(df))
    print('첫 번째 행 price 자료형:', type(df.iloc[0]['price']))
    return df


print('=== 정상 파일 테스트 ===')
df_house = load_house_data(csv_path)

print('\n=== 없는 파일 테스트 ===')
missing_df = load_house_data('missing_house_file.csv')

=== 정상 파일 테스트 ===
첫 번째 행:
ID        1.0
size     34.0
year      5.0
price     5.0
Name: 0, dtype: float64
전체 데이터 행 수: 10
첫 번째 행 price 자료형: <class 'numpy.float64'>

=== 없는 파일 테스트 ===
파일명과 업로드 여부를 확인하세요.


## 1-2. 문자열을 숫자로 변환하기

빈 리스트 `houses`에 `ID`, `size`, `year`는 정수, `price`는 실수로 저장합니다. 누락 또는 변환 오류는 해당 행만 건너뜁니다.

In [2]:
houses = []
excluded_count = 0

if df_house is not None:
    for idx, row in df_house.iterrows():
        row_number = idx + 2  # CSV 헤더를 포함한 행 번호

        # 필수 값 누락: 다른 행의 처리는 계속함
        if any(pd.isna(row[col]) for col in ['ID', 'size', 'year', 'price']):
            print(f'{row_number}행 제외: 필수 값 누락')
            excluded_count += 1
            continue

        try:
            house = {
                'ID': int(row['ID']),
                'size': int(row['size']),
                'year': int(row['year']),
                'price': float(row['price'])
            }
            # 무한대 등 유한하지 않은 숫자는 제외
            if not math.isfinite(house['price']):
                print(f'{row_number}행 제외: 유한하지 않은 가격')
                excluded_count += 1
                continue
            houses.append(house)
        except (ValueError, TypeError, OverflowError) as err:
            print(f'{row_number}행 제외: 숫자 변환 오류 ({err})')
            excluded_count += 1

print('정상 저장 행 수:', len(houses))
print('제외 행 수:', excluded_count)
if houses:
    print('첫 번째 주택:', houses[0])
    print('변환 자료형:', {key: type(value).__name__ for key, value in houses[0].items()})

정상 저장 행 수: 10
제외 행 수: 0
첫 번째 주택: {'ID': 1, 'size': 34, 'year': 5, 'price': 5.0}
변환 자료형: {'ID': 'int', 'size': 'int', 'year': 'int', 'price': 'float'}


## 1-3. 조건에 맞는 주택 찾기

`houses`를 보존한 채 테스트 데이터 4건을 추가합니다. **키 누락 → 결측 → 변환 오류 → 값 범위 → 선택 조건** 순서로 확인합니다.

- 제외: `size <= 0` **or** `price < 0`
- 선택: `size >= 30` **and** `price <= 4`

In [3]:
# 딕셔너리까지 복사하여 원본 houses 유지
houses_test = [house.copy() for house in houses]
houses_test.extend([
    {'ID': 101, 'year': 5, 'price': 3},                 # size 키 없음
    {'ID': 102, 'size': 35, 'year': 5, 'price': None},  # 결측
    {'ID': 103, 'size': '오류', 'year': 5, 'price': 3},  # 변환 불가
    {'ID': 104, 'size': 35, 'year': 5, 'price': 3}     # 정상
])

selected = []

for house in houses_test:
    house_id = house.get('ID', '알 수 없음')

    # 1) 필수 키 존재 여부
    if 'size' not in house or 'price' not in house:
        print(f'ID {house_id} 제외: size 또는 price 키 누락')
        continue

    # 2) None / NaN 여부
    if pd.isna(house['size']) or pd.isna(house['price']):
        print(f'ID {house_id} 제외: size 또는 price 값 누락')
        continue

    # 3) 숫자 변환 및 무한대 검사
    try:
        size = float(house['size'])
        price = float(house['price'])
    except (ValueError, TypeError):
        print(f'ID {house_id} 제외: 숫자로 변환 불가')
        continue

    if not math.isfinite(size) or not math.isfinite(price):
        print(f'ID {house_id} 제외: 유한하지 않은 값')
        continue

    # 4) 유효하지 않은 값은 건너뛰기 (or)
    if size <= 0 or price < 0:
        print(f'ID {house_id} 제외: 크기 또는 가격 범위 오류')
        continue

    # 5) 두 선택 조건을 모두 만족할 때만 추가 (and)
    if size >= 30 and price <= 4:
        selected.append(house)

for house in selected:
    print('선택된 주택 ID:', house['ID'])
print('선택된 주택 전체 건수:', len(selected))
print('원본 houses 건수 유지:', len(houses))

ID 101 제외: size 또는 price 키 누락
ID 102 제외: size 또는 price 값 누락
ID 103 제외: 숫자로 변환 불가
선택된 주택 ID: 3
선택된 주택 ID: 4
선택된 주택 ID: 5
선택된 주택 ID: 9
선택된 주택 ID: 10
선택된 주택 ID: 104
선택된 주택 전체 건수: 6
원본 houses 건수 유지: 10


## 1-4. 함수와 예외 처리

`unit_price(price, size)`는 **가격 ÷ 크기**를 반환합니다. 입력 문제별 안내 문구를 출력하고 `None`을 반환합니다. 전체 주택 적용과 지정된 7개 호출을 모두 실행합니다.

In [4]:
def unit_price(price, size):
    try:
        price = float(price)
        size = float(size)
    except ValueError:
        print('값 오류: 숫자로 변환할 수 없는 문자열입니다.')
        return None
    except TypeError:
        print('자료형 오류: 숫자 또는 숫자 형태의 문자열을 입력하세요.')
        return None

    # 계산 가능한 유한한 숫자인지 확인
    if not math.isfinite(price) or not math.isfinite(size):
        print('값 오류: 유한한 숫자를 입력하세요.')
        return None
    if size <= 0:
        print('크기 오류: 크기는 0보다 커야 합니다.')
        return None
    if price < 0:
        print('가격 오류: 가격은 0 이상이어야 합니다.')
        return None

    return price / size


print('=== 전체 주택의 크기당 가격 ===')
for house in houses:
    result = unit_price(house['price'], house['size'])
    print(f"ID {house['ID']}: {result:.4f}" if result is not None else f"ID {house['ID']}: None")

print('\n=== 지정된 7개 함수 테스트 ===')
test_cases = [
    (4, 40), ('4', '40'), (4, 0), (-4, 40),
    ('오류', 40), (None, 40), (6, 30)
]
for i, (price, size) in enumerate(test_cases, start=1):
    print(f'[{i}] unit_price({price!r}, {size!r})')
    print('반환값:', unit_price(price, size))

=== 전체 주택의 크기당 가격 ===
ID 1: 0.1471
ID 2: 0.1000
ID 3: 0.1333
ID 4: 0.0789
ID 5: 0.0750
ID 6: 0.0875
ID 7: 0.0885
ID 8: 0.1000
ID 9: 0.0882
ID 10: 0.0853

=== 지정된 7개 함수 테스트 ===
[1] unit_price(4, 40)
반환값: 0.1
[2] unit_price('4', '40')
반환값: 0.1
[3] unit_price(4, 0)
크기 오류: 크기는 0보다 커야 합니다.
반환값: None
[4] unit_price(-4, 40)
가격 오류: 가격은 0 이상이어야 합니다.
반환값: None
[5] unit_price('오류', 40)
값 오류: 숫자로 변환할 수 없는 문자열입니다.
반환값: None
[6] unit_price(None, 40)
자료형 오류: 숫자 또는 숫자 형태의 문자열을 입력하세요.
반환값: None
[7] unit_price(6, 30)
반환값: 0.2


## 1-5. 목적에 맞는 자료구조 선택하기

주택 ID의 순서, 연식의 중복 제거, ID별 가격 조회, 변경 불가능한 검색 조건에 적합한 자료구조를 선택합니다.

In [5]:
# 1) 리스트: 입력 순서가 유지되고 중복 ID도 보존됨
# 대안인 집합(set)은 중복을 없애며 순서 보장에 적합하지 않음
house_ids = [house['ID'] for house in houses]
print('전체 주택 ID:', house_ids)

# 2) 집합: 같은 year가 여러 번 나와도 한 번만 저장됨
# 대안인 리스트(list)는 중복을 허용하므로 별도 중복 제거 필요
unique_years = {house['year'] for house in houses}
print('중복 없는 year:', sorted(unique_years))

# 3) 딕셔너리: 주택 ID를 키로 사용해 가격을 바로 조회
# 대안인 리스트(list)는 ID를 기준으로 찾으려면 순회가 필요
price_by_id = {house['ID']: house['price'] for house in houses}
print('ID 3의 가격:', price_by_id[3])

# 4) 튜플: 최소 크기와 최대 가격을 불변 조건으로 저장
# 대안인 리스트(list)는 원소 추가·삭제·수정이 가능함
search_conditions = (30, 4)
print('검색 조건 (최소 크기, 최대 가격):', search_conditions)

전체 주택 ID: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
중복 없는 year: [2, 5, 12, 18, 19, 20, 22]
ID 3의 가격: 4.0
검색 조건 (최소 크기, 최대 가격): (30, 4)


## 제출 전 점검

- [x] 1-1: 정상 파일과 없는 파일의 실행 결과
- [x] 1-2: 빈 리스트 생성, `for` 변환, 정상·제외 행 수
- [x] 1-3: 테스트 4건 추가, `or`·`and`·`continue` 사용, 선택 ID·건수
- [x] 1-4: 함수와 `ValueError`/`TypeError`, 전체 주택 및 테스트 7건
- [x] 1-5: 리스트·집합·딕셔너리·튜플과 각각의 선택 이유·대안 비교

**GitHub 제출:** 이 노트북과 `1.01.house.price.csv`를 개인 저장소에 올리고, 저장소가 Public인지 확인한 다음 LMS에 저장소 URL을 제출하세요.